# dq_engine\nGenerated from 02_processing/dq_engine.py for Databricks notebook import.\n

In [ ]:
"""Data quality engine driven by rule metadata."""\n\nfrom __future__ import annotations\n\n\ndef apply_dq_rules(df, dq_rows: list[dict]):\n    """Apply all DQ rules and annotate each row with status and ALL failing rules.\n\n    A row gets ``dq_status = 'FAIL'`` if it violates *any* rule.  All violated\n    rule names are accumulated in ``dq_failed_rule`` as a comma-separated list\n    so analysts can see every failure, not just the first one discovered.\n    """\n    from pyspark.sql import functions as F\n\n    result = df.withColumn("dq_status", F.lit("PASS")).withColumn("dq_failed_rule", F.lit(None).cast("string"))\n\n    for row in dq_rows:\n        rule_expr = row.get("rule_expression")\n        rule_name = row.get("rule_name", "UNKNOWN")\n        if not rule_expr:\n            continue\n\n        failed_condition = ~F.expr(rule_expr)\n        result = result.withColumn(\n            "dq_status",\n            F.when(failed_condition, F.lit("FAIL")).otherwise(F.col("dq_status")),\n        ).withColumn(\n            # Accumulate ALL failing rule names, not just the first.\n            "dq_failed_rule",\n            F.when(\n                failed_condition,\n                F.when(F.col("dq_failed_rule").isNull(), F.lit(rule_name)).otherwise(\n                    F.concat(F.col("dq_failed_rule"), F.lit(","), F.lit(rule_name))\n                ),\n            ).otherwise(F.col("dq_failed_rule")),\n        )\n\n    return result\n\n\ndef split_valid_reject(df):\n    valid_df = df.filter("dq_status = 'PASS'")\n    reject_df = df.filter("dq_status = 'FAIL'")\n    return valid_df, reject_df\n